# regime 1 (CIFAR-10) / Compare Results — all rules, equal wall-clock
Loads `results/*.json` from Drive (missing ones skipped): backprop, two-factor, three-factor v1/v2/v3, and the 100M ceiling. Loss curves, predictions, cost/memory, cos-sweep head-to-head, summary.

## 1. Setup

In [ ]:
import os, json, math, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_DRIVE, DRIVE_SUBDIR = True, 'Section8_regime1_cifar10'
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive')
        STORE = os.path.join('/content/drive/MyDrive', DRIVE_SUBDIR)
    except Exception as e:
        print('Drive mount failed:', e); STORE = os.path.join('/content', DRIVE_SUBDIR)
else:
    STORE = os.path.join('.', DRIVE_SUBDIR)
RESULTS_DIR = os.path.join(STORE, 'results'); CKPT_DIR = os.path.join(STORE, 'checkpoints')
EXTRA_RESULTS_DIRS = []   # merge results JSONs from other Google accounts here (see the "merge" cell below)
EXTRA_CKPT_DIRS    = []   # extra checkpoint folders for the predictions cell (same idea)
METHODS = ['backprop', 'two_factor', 'three_factor', 'three_factor_v2', 'three_factor_v3', 'backprop_100M']
LABELS  = {'backprop':'Backpropagation', 'two_factor':'Two-factor Hebbian',
           'three_factor':'Three-factor v1 (cos~0.5)', 'three_factor_v2':'Three-factor v2 (cos~0.09)',
           'three_factor_v3':'Three-factor v3 (cos~0.01)', 'backprop_100M':'Backprop 100M (ceiling)'}
COLORS  = {'backprop':'#1F3864', 'two_factor':'#B8860B', 'three_factor':'#C62828',
           'three_factor_v2':'#E67E22', 'three_factor_v3':'#7B1FA2', 'backprop_100M':'#2E7D32'}
print('reading', RESULTS_DIR)

## 1b. Merge results from another account (optional)

In [ ]:
# --- OPTIONAL: merge results/checkpoints exported from ANOTHER Google account ---
# If some methods ran on a different account (their JSONs live on a different Drive), either set
# EXTRA_RESULTS_DIRS above to a Drive folder you can see, OR run this cell and upload the JSON files
# (e.g. three_factor_v3.json). They get merged into the comparison below. Skip if everything's on this Drive.
import os
try:
    from google.colab import files
    up = files.upload()                                  # pick <method>.json (and optionally <method>.pt)
    import shutil
    for fn in up:
        dst = '/content/extra_results' if fn.endswith('.json') else '/content/extra_ckpts'
        os.makedirs(dst, exist_ok=True); shutil.move(fn, os.path.join(dst, fn))
    if os.path.isdir('/content/extra_results') and '/content/extra_results' not in EXTRA_RESULTS_DIRS:
        EXTRA_RESULTS_DIRS.append('/content/extra_results')
    if os.path.isdir('/content/extra_ckpts') and '/content/extra_ckpts' not in EXTRA_CKPT_DIRS:
        EXTRA_CKPT_DIRS.append('/content/extra_ckpts')
    print('EXTRA_RESULTS_DIRS =', EXTRA_RESULTS_DIRS)
except Exception as e:
    print('nothing uploaded (fine if all results are already on this Drive):', e)

## 2. Load Results

In [ ]:
def _find(m, dirs):
    for d in dirs:
        p = os.path.join(d, f'{m}.json')
        if os.path.exists(p): return p
    return None
R = {}; _loaded = []; _missing = []
for m in METHODS:
    p = _find(m, [RESULTS_DIR] + EXTRA_RESULTS_DIRS)
    if p:
        R[m] = json.load(open(p)); _loaded.append(m)
        md, s = R[m]['meta'], R[m]['summary']
        where = '' if os.path.dirname(p) == RESULTS_DIR else f'  <- {os.path.dirname(p)}'
        print(f'loaded  {m:16} {md["total_steps"]:>8,} steps  {md["wall_clock_sec"]/3600:5.2f}h  '
              f'final acc {s["final_acc"]:.3f}  best loss {s["best_loss"]:.4f}{where}')
    else:
        _missing.append(m); print(f'MISSING {m:16} (searched {[RESULTS_DIR] + EXTRA_RESULTS_DIRS})')
print(f'\n=== {len(_loaded)}/{len(METHODS)} loaded: {_loaded}  |  missing: {_missing} ===')

## 3. Configuration

In [ ]:
for m in R:
    md = R[m]['meta']
    print('='*64); print(LABELS[m])
    print(f'  P={md["P"]:,}  seed={md["seed"]}  device={md["device"]}  epochs={md["epochs"]:.3f}')
    print('  config:', json.dumps(md['config']))

## 4. Compare Loss Curves (all methods)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for m in R:
    c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
    ax[0].plot(hrs, c['test_loss'], 'o-', color=COLORS[m], label=LABELS[m])
    ax[1].plot(hrs, c['test_acc'],  'o-', color=COLORS[m], label=LABELS[m])
ax[0].set_xlabel('wall-clock hours'); ax[0].set_ylabel('test loss'); ax[0].set_title('Test loss vs training time'); ax[0].legend()
ax[1].set_xlabel('wall-clock hours'); ax[1].set_ylabel('test accuracy'); ax[1].set_title('Test accuracy vs training time'); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Compare Predictions

In [ ]:
import math, torch
import torch.nn as nn
import torch.nn.functional as F

class MHSA(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        assert dim % heads == 0, "EMBED_DIM must be divisible by HEADS"
        self.h, self.dh = heads, dim // heads
        self.scale = self.dh ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q @ k.transpose(-2, -1)) * self.scale
        att = att.softmax(dim=-1)
        o = (att @ v).transpose(1, 2).reshape(B, N, D)
        return self.proj(o)

class Block(nn.Module):
    def __init__(self, dim, heads, mlp_ratio):
        super().__init__()
        self.n1 = nn.LayerNorm(dim); self.attn = MHSA(dim, heads)
        self.n2 = nn.LayerNorm(dim)
        h = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, h), nn.GELU(), nn.Linear(h, dim))
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.mlp(self.n2(x))
        return x

class ViT(nn.Module):
    def __init__(self, img=28, patch=7, in_ch=1, dim=176, depth=4, heads=8,
                 mlp_ratio=2, num_classes=10):
        super().__init__()
        assert img % patch == 0
        self.n = (img // patch) ** 2
        self.patch = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.zeros(1, self.n + 1, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        nn.init.trunc_normal_(self.cls, std=0.02)
        self.blocks = nn.ModuleList([Block(dim, heads, mlp_ratio) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)
    def forward(self, x):
        B = x.shape[0]
        x = self.patch(x).flatten(2).transpose(1, 2)      # (B, n, dim)
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)[:, 0]
        return self.head(x)

import torchvision
CIFAR_ROOT = globals().get('CIFAR_ROOT', './data'); CIFAR_DL = globals().get('CIFAR_DOWNLOAD', True)
_t = torchvision.datasets.CIFAR10(CIFAR_ROOT, train=False, download=CIFAR_DL)
_mean = torch.tensor([0.4914,0.4822,0.4465]); _std = torch.tensor([0.2470,0.2435,0.2616])
_raw = (torch.as_tensor(_t.data, dtype=torch.float32)/255.0)                  # (N,32,32,3) in [0,1] for display
Xte = ((_raw - _mean)/_std).permute(0,3,1,2).contiguous().to(device)
Yte = torch.as_tensor(_t.targets).to(device)

def _find_ckpt(m):
    for d in [CKPT_DIR] + EXTRA_CKPT_DIRS:
        p = os.path.join(d, f'{m}.pt')
        if os.path.exists(p): return p
    return None

def load_model(m):
    cp = _find_ckpt(m)
    if cp is None: return None
    cfg = R[m]['meta']['config']
    net = ViT(32, cfg['PATCH'], 3, cfg['EMBED_DIM'], cfg['DEPTH'], cfg['HEADS'], cfg['MLP_RATIO'], 10).to(device)
    ck = torch.load(cp, map_location=device)['method']
    net.load_state_dict(ck['net'] if 'net' in ck else ck['params']); net.eval(); return net

_CLS = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']
N = 12; idx = torch.arange(N); preds = {}
for m in R:
    net = load_model(m)
    if net is None:
        print(f'skip predictions for {m}: no checkpoint .pt on this Drive'); continue
    with torch.no_grad(): preds[m] = net(Xte[idx]).argmax(-1).cpu()
if preds:
    fig, axes = plt.subplots(1, N, figsize=(1.6*N, 2.4))
    for j in range(N):
        axes[j].imshow(_raw[idx[j]].numpy()); axes[j].axis('off')
        axes[j].set_title('y=%s\n' % _CLS[Yte[idx[j]].item()] +
                          '\n'.join('%s:%s' % (LABELS[m][:4], _CLS[preds[m][j].item()]) for m in preds), fontsize=6)
    plt.suptitle('Sample test predictions (methods with a checkpoint on this Drive)'); plt.tight_layout(); plt.show()
else:
    print('No checkpoints (.pt) on this Drive for the predictions grid — upload via cell 1b, or skip.')

## 6. Compare Cost & Memory

In [ ]:
def fwd_equiv(m):
    st = R[m]['meta']['total_steps']
    if m == 'three_factor': return st * 2 * R[m]['meta']['config'].get('M', 0)   # antithetic probes
    if m.startswith('backprop'): return st * 3                                   # fwd + ~2x bwd
    return st * 2                                                                # two-factor: acts + measured loss
print(f'{"method":26}{"steps":>10}{"fwd-pass-equiv":>16}{"wall h":>9}{"peak MB":>10}')
print('-'*71)
for m in R:
    md = R[m]['meta']
    print(f'{LABELS[m]:26}{md["total_steps"]:>10,}{fwd_equiv(m):>16,}{md["wall_clock_sec"]/3600:>9.2f}'
          f'{md.get("peak_mem_mb", float("nan")):>10.1f}')

## 7. Head-to-head — three-factor cos sweep (v1/v2/v3)

In [ ]:
# Head-to-head across the three-factor cos sweep: does MORE-but-noisier keep winning in equal wall-clock?
tf = [m for m in ['three_factor', 'three_factor_v2', 'three_factor_v3'] if m in R]
if len(tf) >= 2:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
    for m in tf:
        c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
        ax[0].plot(hrs, c['test_loss'], 'o-', color=COLORS[m], label=LABELS[m])
        ax[1].plot(hrs, c['test_acc'],  'o-', color=COLORS[m], label=LABELS[m])
        ax[2].plot(c['step'], c['test_loss'], 'o-', color=COLORS[m], label=LABELS[m])
    ax[0].set_xlabel('wall-clock hours'); ax[0].set_ylabel('test loss'); ax[0].set_title('three-factor cos sweep — loss vs time'); ax[0].legend()
    ax[1].set_xlabel('wall-clock hours'); ax[1].set_ylabel('test accuracy'); ax[1].set_title('accuracy vs time'); ax[1].legend()
    ax[2].set_xlabel('optimizer steps'); ax[2].set_ylabel('test loss'); ax[2].set_title('loss vs steps'); ax[2].set_xscale('symlog'); ax[2].legend()
    plt.tight_layout(); plt.show()

    print(f'{"variant":30}{"M":>10}{"cos~":>8}{"steps":>10}{"best_loss":>12}{"final_acc":>11}')
    print('-'*81)
    for m in tf:
        md, s = R[m]['meta'], R[m]['summary']
        M = md['config'].get('M') or 0; P = md['P']
        cos = (M/(M+P+1))**0.5 if M else float('nan')
        print(f'{LABELS[m]:30}{M:>10,}{cos:>8.3f}{md["total_steps"]:>10,}{s["best_loss"]:>12.4f}{s["final_acc"]:>11.3f}')
    best = min(tf, key=lambda m: R[m]['summary']['best_loss'])
    print(f'\nLowest best test loss in the same wall-clock budget: {LABELS[best]} '
          f'({R[best]["summary"]["best_loss"]:.4f}). Reading down the cos column shows whether pushing cos '
          f'lower (fewer probes, more steps) keeps helping or finally breaks down.')
else:
    print('Head-to-head needs >=2 three-factor variants present (run at least two of 03 / 05 / 06).')

## 8. Summary table (all methods)

In [ ]:
print(f'{"Experiment":26}{"Epochs":>9}{"First loss":>12}{"Final loss":>12}{"Best loss":>11}{"Reduc %":>9}{"Test acc":>10}')
print('-'*89)
for m in R:
    md, s = R[m]['meta'], R[m]['summary']
    print(f'{LABELS[m]:26}{md["epochs"]:>9.3f}{s["initial_loss"]:>12.4f}{s["final_loss"]:>12.4f}'
          f'{s["best_loss"]:>11.4f}{s["reduction_pct"]:>9.1f}{s["final_acc"]:>10.3f}')
print('\n("Epochs" is fractional: in equal wall-clock, backprop runs many epochs while the three-factor '
      'rule runs a fraction of one -- that gap is the from-scratch barrier.)')